# データ工程: PDFの画像化とローカルVLM（Qwen2.5-VL）による構造化テキスト抽出

このノートブックでは、DGX Sparkのリソースを活用し、以下の処理を行います：
1. 依存パッケージのインストール
2. `row_data` フォルダ内のPDFを画像に変換し、`extract_data` に保存
3. ローカルで動作する最新のVLM（Qwen2.5-VL）を使用して構造化テキストの抽出

## 1. 依存パッケージのインストール

In [25]:
# システム依存パッケージ (poppler-utils) のインストール
!apt-get update && apt-get install -y poppler-utils

# Pythonライブラリのインストール (Qwen2.5-VL対応版)
!pip install pdf2image transformers accelerate qwen-vl-utils ipywidgets python-dotenv pymupdf







Hit:1 http://ports.ubuntu.com/ubuntu-ports noble InRelease
Get:2 http://ports.ubuntu.com/ubuntu-ports noble-updates InRelease [126 kB]
Get:3 http://ports.ubuntu.com/ubuntu-ports noble-backports InRelease [126 kB]
Get:4 http://ports.ubuntu.com/ubuntu-ports noble-security InRelease [126 kB]
Get:5 http://ports.ubuntu.com/ubuntu-ports noble-updates/universe arm64 Packages [1978 kB]
Get:6 http://ports.ubuntu.com/ubuntu-ports noble-updates/main arm64 Packages [2439 kB]
Get:7 http://ports.ubuntu.com/ubuntu-ports noble-security/universe arm64 Packages [1263 kB]
Get:8 http://ports.ubuntu.com/ubuntu-ports noble-security/main arm64 Packages [2081 kB]
Fetched 8140 kB in 3s (2474 kB/s)                        
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (24.02.0-1ubuntu9.8).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. ライブラリのインポートと設定

In [2]:
import torch
import os
from pathlib import Path
from pdf2image import convert_from_path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import IPython.display
from dotenv import load_dotenv

load_dotenv() # .envファイルを読み込む

# パスの設定
current_dir = Path(os.getcwd())
print(f"Current working directory: {current_dir}")

# row_data フォルダがノートブックの場所にあると想定されるため、探索パスを定義します
search_paths = [
    # 1. コンテナ内での絶対パスの候補
    Path("/workspace/01_training/01_data_prep"),
    # 2. カレントディレクトリの隣にある 01_training を探す場合（/workspace/container にいる場合）
    current_dir.parent / "01_training" / "01_data_prep",
    # 3. カレントディレクトリ内の場合
    current_dir
]

BASE_DIR = None
for p in search_paths:
    if (p / "row_data").exists():
        BASE_DIR = p
        print(f"Found row_data in: {BASE_DIR.absolute()}")
        break

if BASE_DIR is None:
    print("Warning: 'row_data' directory not found in typical locations. Using Current working directory.")
    BASE_DIR = current_dir

ROW_DATA_DIR = BASE_DIR / "row_data"
EXTRACT_DATA_DIR = BASE_DIR / "extract_data"

print(f"ROW_DATA_DIR: {ROW_DATA_DIR.absolute()}")
print(f"EXTRACT_DATA_DIR: {EXTRACT_DATA_DIR.absolute()}")

EXTRACT_DATA_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Current working directory: /workspace
Found row_data in: /workspace/01_training/01_data_prep
ROW_DATA_DIR: /workspace/01_training/01_data_prep/row_data
EXTRACT_DATA_DIR: /workspace/01_training/01_data_prep/extract_data
Using device: cuda


## 2.2 Huggingfaceへのログイン

In [3]:
from huggingface_hub import login
# これを実行すると、トークンを入力するテキストボックスが表示されます
login()


## 3. PDFから画像への変換

### 3.1 関数定義

In [4]:
def convert_pdfs_to_images(row_data_path, output_path):
    """
    row_data_path 内のすべてのPDFを全ページ画像化します。
    """
    row_data_path = Path(row_data_path)
    output_path = Path(output_path)
    
    # Linux環境の大文字小文字を考慮して探索 (.pdf と .PDF)
    pdf_files = sorted(list(row_data_path.glob("*.pdf")) + list(row_data_path.glob("*.PDF")))
    
    print(f"Searching in: {row_data_path.absolute()}")
    print(f"Found {len(pdf_files)} PDF files.")
    
    if not pdf_files:
        # フォルダが存在するか確認し、存在する場合は中身をデバッグ表示
        if row_data_path.exists():
            print(f"Directory exists, files in it: {[f.name for f in row_data_path.iterdir()]}")
        else:
            print(f"Error: Directory {row_data_path} does not exist.")
        return
    
    for pdf_file in pdf_files:
        print(f"Converting {pdf_file.name}...")
        try:
            images = convert_from_path(str(pdf_file))
            for i, image in enumerate(images):
                image_name = f"{pdf_file.stem}_page_{i+1:03d}.png"
                image.save(output_path / image_name, "PNG")
                print(f" Saved: {image_name}")
        except Exception as e:
            print(f" Error converting {pdf_file.name}: {e}")

### 3.2 実行

In [ ]:
convert_pdfs_to_images(ROW_DATA_DIR, EXTRACT_DATA_DIR)

## 4. ローカルVLM（Qwen2.5-VL）によるテキスト抽出

### 4.1 関数定義

In [13]:
def load_model(model_name="Qwen/Qwen2.5-VL-7B-Instruct"):
    """
    Qwen2.5-VL モデルとプロセッサをロードします。
    """
    print(f"Loading model {model_name}...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_name)
    return model, processor

def extract_text_locally(image_path, model, processor, prompt):
    """
    ローカルのQwen2.5-VLを使用して画像から構造化テキストを抽出します。
    """
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # プロンプトの準備
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(device)

    # 推論の実行
    generated_ids = model.generate(**inputs, max_new_tokens=4096)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

### 4.2 モデルのロード

In [8]:
# モデルのロード（初回実行時はダウンロードに時間がかかります）
model, processor = load_model()

Loading model Qwen/Qwen2.5-VL-7B-Instruct...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

### 4.3 推論の実行

In [21]:
prompt = """
この画像のテキストを、レイアウト（見出し、表、箇条書き）を可能な限り保持してマークダウン形式で抽出してください。
「この形式でテキストが抽出されました。」の文言は不要です。
"""




In [22]:
# 最初の1ページを処理
sample_images = sorted(list(EXTRACT_DATA_DIR.glob("*.png")))
if sample_images:
    sample_path = sample_images[13]
    print(f"Processing: {sample_path.name}")
    extracted_content = extract_text_locally(sample_path, model, processor, prompt)
    print("\n--- Extracted Content ---")
    print(extracted_content)
else:
    print("No images found. Please run Step 3 (Execution) first.")

Processing: r7syogaisyahukusinotebiki_shusei_page_014.png

--- Extracted Content ---
```markdown
# 障害程度別対象事業一覧表 (〇対象・△一部対象)

| 障害の種別 | 本 文 頁 | 所 得 制 限 | 身 体 障 害 者 手 帳 |
| --- | --- | --- | --- |
|  |  |  | 視覚障害 | 聴覚又は平衡感覚機能障害 |
|  |  |  | 1 | 2 | 3 | 4 | 2 | 3 | 4 |
| 事业 |  |  |  |  |  |  |  |  |  |
| 東京メトロ旅客運賃の割引 | 120 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| 私鉄旅客運賃の割引 | 120 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| タクシー運賃の割引 | 120 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| 航空運賃の割引 | 121 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| 旅客船運賃の割引 | 121 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| テレビ受信料の減免 | 121 | 有 | △ | △ | △ | △ | △ | △ | △ |
| 水道・下水道料金の減免 | 122 | 有 | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 |
| 携帯電話使用料等の割引 | 123 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| 郵便料金・ゆうパック運賃等の減免 | 123 |  | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 | 本文参照 |
| 通常郵便葉書(青い鳥郵便葉書)の無料配布 | 124 |  | 〇 | 〇 |  |  |  | 〇 |  |
| 税の軽減 |  |  |  |  |  |  |  |  |  |
| 所得税・住民税の障害者控除 | 125 |  | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 | 〇 |
| 住民税の非課税 | 126 | 有 | 〇 | 〇 | 〇 | 〇 

### 4.3 全ファイルバッチ推論

In [24]:
import time

# Create directory to save extracted texts
EXTRACTED_TEXTS_DIR = BASE_DIR / "extracted_texts"
EXTRACTED_TEXTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {EXTRACTED_TEXTS_DIR}")

# Get all converted images
all_images = sorted(list(EXTRACT_DATA_DIR.glob("*.png")))
total_images = len(all_images)
print(f"Found {total_images} images to process.")

if total_images > 0:
    total_start_time = time.time()
    
    for i, image_path in enumerate(all_images):
        output_file = EXTRACTED_TEXTS_DIR / f"{image_path.stem}.md"
        
        # Skip if already extracted
        if output_file.exists():
            print(f"[{i+1}/{total_images}] Skipping (already exists): {output_file.name}")
            continue
            
        print(f"[{i+1}/{total_images}] Processing: {image_path.name} ...")
        
        try:
            start_time = time.time() # Start time per file
            
            # Execute text extraction
            extracted_content = extract_text_locally(image_path, model, processor, prompt)
            
            # Save as Markdown file
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(extracted_content)
                
            elapsed_time = time.time() - start_time # Time taken per file
            
            # Output processing time clearly
            print(f"  -> Time taken: {elapsed_time:.2f}s (Saved to {output_file.name})")
            
        except Exception as e:
            print(f"  -> Error processing {image_path.name}: {e}")
            
    total_elapsed_time = time.time() - total_start_time # Total time
    print(f"\nAll processing completed! Total time taken: {total_elapsed_time:.2f}s")
    
else:
    print("No images found to process. Please check your EXTRACT_DATA_DIR.")


Output directory: /workspace/01_training/01_data_prep/extracted_texts
Found 218 images to process.
[1/218] Processing: r7syogaisyahukusinotebiki_shusei_page_001.png ...
  -> Time taken: 61.52s (Saved to r7syogaisyahukusinotebiki_shusei_page_001.md)
[2/218] Processing: r7syogaisyahukusinotebiki_shusei_page_002.png ...
  -> Time taken: 161.17s (Saved to r7syogaisyahukusinotebiki_shusei_page_002.md)
[3/218] Processing: r7syogaisyahukusinotebiki_shusei_page_003.png ...
  -> Time taken: 50.14s (Saved to r7syogaisyahukusinotebiki_shusei_page_003.md)
[4/218] Processing: r7syogaisyahukusinotebiki_shusei_page_004.png ...
  -> Time taken: 69.12s (Saved to r7syogaisyahukusinotebiki_shusei_page_004.md)
[5/218] Processing: r7syogaisyahukusinotebiki_shusei_page_005.png ...
  -> Time taken: 79.79s (Saved to r7syogaisyahukusinotebiki_shusei_page_005.md)
[6/218] Processing: r7syogaisyahukusinotebiki_shusei_page_006.png ...
  -> Time taken: 78.25s (Saved to r7syogaisyahukusinotebiki_shusei_page_006.md)


### 4.4 pymupdfによるテキスト抽出

In [26]:
# ---------------------------------------------------------
# 5. [Alternative] Pythonライブラリ(PyMuPDF)による高速テキスト抽出（ページ分割版）
# ---------------------------------------------------------

# ライブラリのインストール
!pip install pymupdf

import time
import fitz  # PyMuPDF
from pathlib import Path

# テキスト保存先ディレクトリの作成（VLM抽出結果とは別フォルダに保存します）
EXTRACTED_PY_DIR = BASE_DIR / "extracted_texts_python"
EXTRACTED_PY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory (Python Library): {EXTRACTED_PY_DIR}")

# 処理対象の元PDFファイルを取得
pdf_files = sorted(list(ROW_DATA_DIR.glob("*.pdf")) + list(ROW_DATA_DIR.glob("*.PDF")))
total_pdfs = len(pdf_files)
print(f"Found {total_pdfs} PDF files to process.")

if total_pdfs > 0:
    total_start_time = time.time()
    
    for pdf_path in pdf_files:
        print(f"Processing PDF: {pdf_path.name} ...")
        
        try:
            pdf_start_time = time.time()
            
            # PDFを開いてページごとに処理
            with fitz.open(pdf_path) as doc:
                total_pages = len(doc)
                
                for page_num in range(total_pages):
                    # ページ番号を3桁（001, 002...）でフォーマット
                    page_str = f"{page_num + 1:03d}"
                    output_file = EXTRACTED_PY_DIR / f"{pdf_path.stem}_page_{page_str}.md"
                    
                    if output_file.exists():
                        print(f"  [{page_num+1}/{total_pages}] Skipping: {output_file.name}")
                        continue
                        
                    page = doc[page_num]
                    # PDFに埋め込まれているテキストを抽出
                    text = page.get_text("text") 
                    
                    # Markdownファイルとして保存
                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write(text.strip())
                        
                    print(f"  [{page_num+1}/{total_pages}] Saved: {output_file.name}")
                    
            elapsed_time = time.time() - pdf_start_time
            print(f"-> Finished {pdf_path.name} in {elapsed_time:.2f}s\n")
            
        except Exception as e:
            print(f"-> Error processing {pdf_path.name}: {e}\n")
            
    total_elapsed_time = time.time() - total_start_time
    print(f"All processing completed! Total time taken: {total_elapsed_time:.2f}s")
else:
    print("No PDFs found to process in ROW_DATA_DIR.")


Output directory (Python Library): /workspace/01_training/01_data_prep/extracted_texts_python
Found 1 PDF files to process.
Processing PDF: r7syogaisyahukusinotebiki_shusei.pdf ...
  [1/218] Saved: r7syogaisyahukusinotebiki_shusei_page_001.md
  [2/218] Saved: r7syogaisyahukusinotebiki_shusei_page_002.md
  [3/218] Saved: r7syogaisyahukusinotebiki_shusei_page_003.md
  [4/218] Saved: r7syogaisyahukusinotebiki_shusei_page_004.md
  [5/218] Saved: r7syogaisyahukusinotebiki_shusei_page_005.md
  [6/218] Saved: r7syogaisyahukusinotebiki_shusei_page_006.md
  [7/218] Saved: r7syogaisyahukusinotebiki_shusei_page_007.md
  [8/218] Saved: r7syogaisyahukusinotebiki_shusei_page_008.md
  [9/218] Saved: r7syogaisyahukusinotebiki_shusei_page_009.md
  [10/218] Saved: r7syogaisyahukusinotebiki_shusei_page_010.md
  [11/218] Saved: r7syogaisyahukusinotebiki_shusei_page_011.md
  [12/218] Saved: r7syogaisyahukusinotebiki_shusei_page_012.md
  [13/218] Saved: r7syogaisyahukusinotebiki_shusei_page_013.md
  [14/218